# Nigerian Language Identification: Live Demo

Fine-tuned **WavLM Base Plus** that identifies which Nigerian language is being spoken.
Model: [AbdulroqeebA/wavlm-base-plus-nigerian-lid](https://huggingface.co/AbdulroqeebA/wavlm-base-plus-nigerian-lid)

Built at the DSN AI Bootcamp 2026 hackathon.

**How to use:** run the cells below one after another (Runtime → Run all). When the last cell finishes, click the public `gradio.live` link it prints, then upload or record a short speech clip.

No GPU is needed, and no token is needed because the model is public.

In [ ]:
# 1. Install what Colab does not already have
!pip -q install gradio librosa soundfile

In [ ]:
# 2. Load the model from Hugging Face
import numpy as np
import torch
import librosa
import gradio as gr
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

MODEL_ID = "AbdulroqeebA/wavlm-base-plus-nigerian-lid"
TARGET_SR = 16000
MAX_DURATION_SEC = 10.0
MAX_LEN = int(MAX_DURATION_SEC * TARGET_SR)

# Display names for label codes. Check id2label printed below and add any missing codes.
LANG_NAMES = {
    "yor": "Yoruba",
    "ibo": "Igbo",
    "hau": "Hausa",
    "efi": "Efik",
    "ibb": "Ibibio",
    "igl": "Igala",
    "nup": "Nupe",
    "other": "Other",
}

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModelForAudioClassification.from_pretrained(MODEL_ID)
model = model.float().eval()  # weights were saved in FP16; run in FP32
id2label = model.config.id2label
print("Labels:", id2label)

In [ ]:
# 3. Prediction function (same preprocessing as training)
def predict(audio_path):
    if audio_path is None:
        return {}

    audio, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    audio = np.asarray(audio, dtype=np.float32)
    if len(audio) == 0:
        return {}

    # Centre-crop clips longer than 10 s, as done at validation time
    if len(audio) > MAX_LEN:
        start = (len(audio) - MAX_LEN) // 2
        audio = audio[start:start + MAX_LEN]

    inputs = feature_extractor(
        audio,
        sampling_rate=TARGET_SR,
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    with torch.no_grad():
        logits = model(**inputs).logits[0]
    probs = torch.softmax(logits, dim=-1).numpy()

    return {LANG_NAMES.get(id2label[i], id2label[i]): float(p) for i, p in enumerate(probs)}

In [ ]:
# 4. Launch the demo. Click the public gradio.live link it prints.
demo = gr.Interface(
    fn=predict,
    inputs=gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
        label="Upload or record speech (up to 10 seconds is used)",
    ),
    outputs=gr.Label(num_top_classes=3, label="Top 3 predicted languages"),
    title="Nigerian Language Identification",
    description=(
        "Fine-tuned WavLM Base Plus that identifies which Nigerian language is being spoken. "
        "Validation macro F1 is about 0.59, so treat results as a guide, not a certainty. "
        "Short, clear speech works best."
    ),
    flagging_mode="never",
)

demo.launch(share=True)

**Notes**
- The public link works for about 72 hours while this notebook keeps running, then it stops.
- Results are a guide only: validation macro F1 is about 0.59 (0.62 after bias calibration).
- Credits: DSN AI Bootcamp 2026 hackathon team. Dataset: `Data-Science-Nigeria/dsn-2026-lid-train` (mixed per-source licenses, see the dataset page).